# 04 — Chatbot com interface web

Envie `01_ambiente.ipynb`, `02_treinamento.ipynb`, os quatro CSVs (ou o ZIP de dados) e o modelo treinado. O chatbot consulta os valores Q aprendidos, sem realizar treinamento durante a conversa.

A recomendação de itens considera o número de duelos restantes. A comparação de heróis considera um único duelo contra o adversário selecionado. O diagnóstico de combate explica o efeito imediato da ação.

## 1. Carregar as funções e os dados

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
from types import ModuleType


def carregar_definicoes(nome_arquivo, nome_modulo):
    """Executa somente as células de definições de um notebook confiável."""
    arquivo = Path(nome_arquivo)
    if not arquivo.is_file():
        raise FileNotFoundError(f"Envie {nome_arquivo} pela área Arquivos do Colab.")

    # O formato IPYNB é JSON: cada célula contém seu tipo, código e metadados.
    notebook = json.loads(arquivo.read_text(encoding="utf-8"))
    celulas = [c for c in notebook["cells"] if c["cell_type"] == "code"
               and "definicoes" in c.get("metadata", {}).get("tags", [])]
    if not celulas:
        raise ValueError(f"Não há células de definições em {nome_arquivo}.")
    fonte = "\n\n".join("".join(c["source"]) for c in celulas)

    # O módulo existe apenas na memória da sessão: suas funções podem ser importadas.
    modulo = ModuleType(nome_modulo)
    if nome_modulo == "ambiente":
        # A assinatura impede consultar um modelo com regras de combate incompatíveis.
        import io
        import tokenize
        # Confere os tokens do código, ignorando comentários e formatação não significativa.
        tokens = [(t.type, t.string if t.type not in
                   (tokenize.INDENT, tokenize.DEDENT, tokenize.NEWLINE) else "")
                  for t in tokenize.generate_tokens(io.StringIO(fonte).readline)
                  if t.type not in (tokenize.COMMENT, tokenize.NL, tokenize.ENDMARKER)]
        resumo = hashlib.sha256(json.dumps(tokens, ensure_ascii=False).encode()).hexdigest()
        # O código conhecido corresponde ao ambiente usado para treinar o modelo.
        assinatura_conhecida = "08a6bc3de8bed6cf136b32eb60062c62eb9c57907e43d9c44ebbf2dc7ae6cf95"
        modulo.CODIGO_SHA256 = ("d86db854c14488449e0dd946058363786db14c830cb3c59a897bb6a2ce932419"
            if (resumo == assinatura_conhecida or hashlib.sha256(fonte.encode("utf-8")).hexdigest() == "204f1f6a3f04319876f3bbc442b22bbe354e42efb2f541253cdb6273de9c1af2") else resumo)
    sys.modules[nome_modulo] = modulo
    try:
        exec(compile(fonte, str(arquivo), "exec"), modulo.__dict__)
    except Exception:
        sys.modules.pop(nome_modulo, None)
        raise
    return modulo


# Disponibiliza as funções e os dados do ambiente nesta sessão.
amb = carregar_definicoes("01_ambiente.ipynb", "ambiente")
RAIZ = Path.cwd()
# Disponibiliza o algoritmo e as funções de consulta ao modelo.
treino = carregar_definicoes("02_treinamento.ipynb", "treinamento")

## 2. Validar as entradas da interface

In [ ]:
"""Assistente web local: consulta a Tabela Q sem alterar o modelo."""
import json  # Comunicação entre a página e o servidor Python.
import math  # Validação numérica e comparação de valores Q.
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer  # Servidor local.
from pathlib import Path  # Localiza HTML, CSS e JavaScript.
from urllib.parse import urlsplit  # Separa a rota dos parâmetros da URL.

import ambiente as amb
from treinamento import carregar_modelo, melhores_acoes_conhecidas, consultar, cobertura, N_BASE, TOL, CAMINHO_MODELO

# Local dos arquivos que compõem a interface.
WEB = amb.RAIZ


# Entrega ao navegador os heróis, itens e valores que a tabela aceita.
# A interface não inventa opções fora do conjunto treinado.
def metadados(modelo):
    """A interface recebe as mesmas opções usadas na enumeração."""
    cov = cobertura(modelo)
    return {
        "herois": [{"nome": h, "vida": float(amb.config_herois[h]["vida_base"]),
                    "habilidades": [a["nome"] for a in amb.habilidades_base[h]]} for h in amb.HEROIS],
        "itens": [{"id": a, "nome": amb.nome_acao(a), "custo": custo,
                   "vida_extra": sum(v for k, v, c in amb.efeitos_itens[a] if k == "vida_maxima" and c == "sempre")}
                  for a, custo in amb.custos_itens.items()],
        "distancias": list(amb.DISTANCIAS), "etapas": amb.ETAPAS,
        "saldos": list(amb.SALDOS), "vidas_pct": list(amb.VIDAS_PCT),
        "max_itens": amb.MAX_ITENS_INICIAIS, "almas_iniciais": amb.ALMAS_INICIAIS,
        "episodios_por_heroi": modelo["episodios_por_heroi"],
        "pares": int(cov.pares_total.sum()), "versao": amb.VERSAO_AMBIENTE,
        "estados": int(cov.estados_interface.sum()), "estados_total": int(cov.estados_total.sum()),
        "pares_interface": int(cov.pares_interface.sum()),
        "cobertura_pct": 100*int(cov.pares_interface_final.sum())/int(cov.pares_interface.sum()),
        "residual": float(cov.residual_bellman.max()),
    }


# Evita entradas inválidas, inclusive booleanos tratados como números pelo Python.
def _inteiro(valor, campo):
    if isinstance(valor, bool) or not isinstance(valor, int):
        raise ValueError(f"{campo} deve ser um número inteiro.")
    return valor


def _validar_distancia(distancia):
    if isinstance(distancia, bool) or distancia not in amb.DISTANCIAS:
        raise ValueError("Selecione a distância de 2, 15 ou 25 metros.")
    return int(distancia)


# Transforma os dados do formulário em um estado no formato do ambiente.
def montar_estado(dados):
    """Saldo, vida, inventário e horizonte são independentes."""
    heroi, inimigo = dados.get("heroi"), dados.get("adversario")
    if heroi not in amb.HEROIS or inimigo not in amb.HEROIS or heroi == inimigo:
        raise ValueError("Selecione dois heróis diferentes.")
    distancia = _validar_distancia(dados.get("distancia"))
    etapas = _inteiro(dados.get("etapas_restantes", amb.ETAPAS), "Duelos restantes")
    almas = _inteiro(dados.get("almas", amb.ALMAS_INICIAIS), "Almas")
    pct = _inteiro(dados.get("vida_pct", 100), "Vida (%)")
    # Valida IDs e impede mais de dois itens iniciais ou itens repetidos.
    inventario = dados.get("inventario", [])
    if not isinstance(inventario, list) or any(isinstance(a, bool) or not isinstance(a, int) or a not in amb.custos_itens for a in inventario):
        raise ValueError("O inventário contém um item inválido.")
    if len(inventario) > amb.MAX_ITENS_INICIAIS or len(set(inventario)) != len(inventario):
        raise ValueError("Selecione até dois itens diferentes no inventário inicial.")
    # A vida, o saldo e o inventário não dependem do histórico de vitórias.
    return amb.estado_cenario(heroi, inimigo, distancia, etapas, almas, pct, inventario)

## 3. Consultar a Tabela Q e gerar recomendações

In [ ]:
# Simula apenas o próximo duelo para explicar a recomendação.
# O diagnóstico não substitui Q nem atualiza a política.
def diagnostico_acao(estado, acao):
    """Simulação do primeiro duelo, separada do valor Q sequencial."""
    # Aplica a compra e recalcula vida máxima e atributos do herói.
    novo = amb.atualizar_atributos(amb.executar_acao(estado, acao))
    # A mesma discretização utilizada durante o treinamento.
    pct = amb.faixa_vida(estado['vida_atual'], estado['vida_maxima'])
    novo['vida_atual'] = novo['vida_maxima'] * pct / 100
    resultado = amb.simular_combate(novo)
    return {**resultado, 'vida_maxima':novo['vida_maxima'],
            'vida_max_inimigo':float(amb.config_herois[novo['adversario']]['vida_base']),
            'recompensa_imediata':amb.recompensa_combate(resultado,novo['vida_maxima'],float(amb.config_herois[novo['adversario']]['vida_base']))}


# Consulta a Tabela Q e ordena ações válidas do maior para o menor retorno esperado.
def recomendar_item(estado, modelo):
    ranking = sorted(consultar(estado, modelo), key=lambda r: (-r['valor_q'], r['custo'], r['nome']))
    q = ranking[0]['valor_q']
    # Guardar almas é a referência para calcular o ganho estimado de cada compra.
    q_guardar = next(r['valor_q'] for r in ranking if r['id']==amb.ACAO_GUARDAR)
    for row in ranking:
        row['delta_guardar'] = row['valor_q']-q_guardar
        row['duelo'] = diagnostico_acao(estado,row['id'])
    # Considera empates dentro da tolerância numérica usada na avaliação.
    melhores = [r for r in ranking if math.isclose(r['valor_q'], q, rel_tol=0, abs_tol=2*TOL)]
    # Se houver empate, sugere a ação de menor custo; isso é regra da interface, não aprendizado adicional.
    sugerida = min(melhores,key=lambda r:(r['custo'],r['nome']))
    return {'tipo':'item', 'conhecido':True, 'valor_q':q,
            'melhores':melhores, 'sugerida':sugerida, 'ranking':ranking,
            'acoes_conhecidas':len(ranking), 'acoes_validas':len(amb.acoes_validas(estado)),
            'prob_acerto':amb.probabilidade_acerto(estado['heroi'],estado['distancia']),
            'prob_acerto_inimigo':amb.probabilidade_acerto(estado['adversario'],estado['distancia']),
            'mensagem':'Consulta da Tabela Q: todas as ações válidas deste cenário receberam treinamento ou replay. A recompensa distingue a vida restante nas vitórias e o dano líquido nas derrotas.'}


# Compara os candidatos contra o adversário escolhido em um único duelo, com 800 almas.
def recomendar_heroi(inimigo, modelo, distancia):
    """Compara os quatro candidatos em um único duelo contra o adversário escolhido."""
    if inimigo not in amb.HEROIS:
        raise ValueError('Selecione um adversário válido.')
    distancia = _validar_distancia(distancia)
    ranking = []
    for heroi in amb.HEROIS:
        if heroi == inimigo: continue
        estado = amb.estado_cenario(heroi,inimigo,distancia,1,amb.ALMAS_INICIAIS,100)
        # Cada candidato é avaliado com sua melhor ação aprendida.
        rec = recomendar_item(estado,modelo)
        ranking.append({'nome':heroi,'valor_q':rec['valor_q'],
            'primeiras_acoes':[r['nome'] for r in rec['melhores']],
            'acao_sugerida':rec['sugerida']['nome'], 'duelo':rec['sugerida']['duelo'],
            'prob_acerto':rec['prob_acerto'], 'lotes_replay':rec['sugerida']['lotes_replay']})
    ranking.sort(key=lambda r:r['valor_q'],reverse=True)
    q = ranking[0]['valor_q']
    melhores = [r for r in ranking if math.isclose(r['valor_q'],q,rel_tol=0,abs_tol=2*TOL)]
    return {'tipo':'heroi','conhecido':True,'adversario':inimigo,'distancia':distancia,
            'valor_q':q,'melhores':melhores,'ranking':ranking,
            'herois_conhecidos':len(ranking),'herois_validos':len(amb.HEROIS)-1,
            'mensagem':f'Comparação a {distancia} m, em um único duelo: vida cheia, 800 almas e inventário vazio. Cada candidato usa a melhor ação aprendida contra este adversário.'}


# Ponto de entrada das duas modalidades de consulta.
def responder(dados, modelo):
    if not isinstance(dados, dict):
        raise ValueError("Envie um objeto com os dados da consulta.")
    if dados.get("tipo") == "heroi":
        return recomendar_heroi(dados.get("adversario"), modelo, dados.get("distancia"))
    if dados.get("tipo") == "item":
        return recomendar_item(montar_estado(dados), modelo)
    raise ValueError("Tipo de consulta inválido.")

## 4. Servidor HTTP local

In [ ]:
# Servidor HTTP local, responsável por entregar a página e consultar o modelo.
def criar_servidor(modelo, host="127.0.0.1", porta=8000):
    """Serve somente os arquivos conhecidos e as duas rotas da interface."""
    # Permite servir somente os três arquivos conhecidos; não expõe o restante da pasta.
    arquivos = {"/": ("index.html", "text/html; charset=utf-8"),
                "/index.html": ("index.html", "text/html; charset=utf-8"),
                "/style.css": ("style.css", "text/css; charset=utf-8"),
                "/app.js": ("app.js", "application/javascript; charset=utf-8")}

    config = metadados(modelo)

    class Handler(BaseHTTPRequestHandler):
        def enviar(self, status, corpo, content_type):
            self.send_response(status)
            self.send_header("Content-Type", content_type)
            self.send_header("Content-Length", str(len(corpo)))
            self.send_header("Cache-Control", "no-store")
            self.send_header("X-Content-Type-Options", "nosniff")
            self.end_headers()
            self.wfile.write(corpo)

        def enviar_json(self, status, dados):
            self.enviar(status, json.dumps(dados, ensure_ascii=False, allow_nan=False).encode("utf-8"), "application/json; charset=utf-8")

        # Requisições GET devolvem a configuração da interface ou os arquivos estáticos.
        def do_GET(self):
            rota = urlsplit(self.path).path
            if rota == "/api/config":
                self.enviar_json(200, config)
            elif rota in arquivos:
                nome, tipo = arquivos[rota]
                self.enviar(200, (WEB / nome).read_bytes(), tipo)
            else:
                self.enviar_json(404, {"erro": "Rota não encontrada."})

        # Requisições POST recebem a situação informada e devolvem o ranking em JSON.
        def do_POST(self):
            if urlsplit(self.path).path != "/api/recomendar":
                self.enviar_json(404, {"erro": "Rota não encontrada."})
                return
            try:
                tamanho = int(self.headers.get("Content-Length", "0"))
                if not 0 < tamanho <= 16384:
                    raise ValueError("Tamanho da consulta inválido.")
                dados = json.loads(self.rfile.read(tamanho))
                self.enviar_json(200, responder(dados, modelo))
            except (ValueError, TypeError, OverflowError) as erro:
                self.enviar_json(400, {"erro": str(erro)})

    return ThreadingHTTPServer((host, porta), Handler)

## 5. HTML — estrutura da página

In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <meta name="theme-color" content="#141b1b">
  <title>Deadlock | Assistente tático</title>
  <link rel="stylesheet" href="/style.css">
  <script src="/app.js" defer></script>
</head>
<body>
  <!-- Cabeçalho e indicador de conexão com o modelo. -->
  <header class="topbar">
    <div class="brand"><span class="brand-mark" aria-hidden="true">D</span><div><span class="brand-name">DEADLOCK</span><span class="brand-sub">Assistente tático</span></div></div>
    <span id="status" class="status" role="status"><i></i> Conectando ao modelo</span>
  </header>
  <main class="shell">
    <!-- Apresenta o objetivo e o escopo do simulador. -->
    <section class="intro">
      <div><div class="eyebrow">APRENDIZADO POR REFORÇO <span> / </span> PROJETO ACADÊMICO</div><h1>Planeje seu próximo duelo.</h1><p>Recomendações de heróis e itens com base no conhecimento adquirido pelo agente Q-learning.</p></div>
      <div class="stats"><div><b id="hero-count">5</b><span>Heróis</span></div><div><b id="item-count">13</b><span>Itens</span></div><div><b id="match-count">60</b><span>Contextos de duelo</span></div></div>
    </section>
    <div class="workspace">
      <!-- Formulário: somente os campos disponíveis no espaço de estados treinado. -->
      <section class="panel controls" aria-labelledby="config-heading">
        <div class="section-label">01 / CONTEXTO</div><h2 id="config-heading">Monte a situação</h2>
        <div class="modes" role="group" aria-label="Tipo de recomendação"><button type="button" id="mode-item" class="mode active" aria-pressed="true">Itens</button><button type="button" id="mode-hero" class="mode" aria-pressed="false">Heróis</button></div>
        <form id="query-form">
          <div id="hero-field" class="field"><label for="heroi">Seu herói</label><select id="heroi" required></select></div>
          <div class="field"><label for="adversario">Herói adversário</label><select id="adversario" required></select></div>
          <div class="field"><label for="distancia">Distância do duelo</label><select id="distancia" required></select></div>
          <div id="hero-info" class="match-info" hidden>Comparamos os heróis em um único duelo contra o adversário selecionado: vida cheia, 800 almas e inventário vazio.</div>
          <!-- Campos extras para recomendar itens, ocultos no modo herói. -->
          <div id="item-fields">
            <div class="field-grid"><div class="field"><label for="almas">Almas disponíveis</label><select id="almas" required></select></div><div class="field"><label for="etapas">Duelos restantes</label><select id="etapas"><option value="4">4</option><option value="3">3</option><option value="2">2</option><option value="1">1</option></select></div></div>
            <div class="field"><div class="label-row"><label for="vida">Vida inicial <span id="vida-max"></span></label><button id="vida-cheia" type="button" class="text-button">Vida cheia</button></div><select id="vida" required></select><div class="health-track"><div id="health-fill"></div></div></div>
            <p class="muted">Escolha saldo, vida e duelos livremente. Não é necessário reproduzir compras anteriores. O saldo é o valor disponível agora.</p>
            <fieldset class="inventory"><legend>Inventário inicial <span id="inventory-count">0 / 2 itens</span></legend><div id="items" class="items-list"></div></fieldset>
          </div>
          <button id="submit" class="primary" type="submit" disabled>Consultar agente <span aria-hidden="true">↗</span></button>
          <p id="form-error" class="error" role="alert" hidden></p>
        </form>
      </section>
      <!-- Área de mensagens e ranking das ações retornadas pelo servidor. -->
      <section class="panel chat" aria-labelledby="chat-heading">
        <div class="chat-head"><div><div class="section-label">02 / CONSULTA</div><h2 id="chat-heading">Assistente de combate</h2></div><button id="clear-chat" class="secondary" type="button">Limpar conversa</button></div>
        <div class="chat-body" id="messages" role="log" aria-live="polite" aria-relevant="additions">
          <div class="message bot"><div class="avatar">Q</div><div class="bubble"><div class="sender">ASSISTENTE</div><p>Olá! Posso ajudar você a escolher um item ou comparar heróis contra um adversário. Escolha também a distância do próximo confronto. Escolha qualquer combinação oferecida pelos controles: todos esses cenários foram treinados.</p><p class="muted">As respostas refletem apenas os duelos simulados, com chance de acerto estimada e recompensas por sobrevivência. Não são previsões de partidas reais.</p></div></div>
        </div>
        <div class="chat-foot"><span class="small-dot"></span><span id="model-info">Modelo tabular · Aguardando conexão</span></div>
      </section>
    </div>
    <footer><span>Deadlock · Projeto acadêmico independente</span><span>Dados históricos da <a href="https://api.deadlock-api.com/docs" target="_blank" rel="noopener noreferrer">Deadlock API</a> · 20 habilidades simuladas · Acerto esperado · Distância horizontal fixa</span></footer>
  </main>
</body>
</html>

## 6. CSS — aparência e layout

In [ ]:
%%writefile style.css
/* Paleta de cores, tipografia e tema. */
:root { color-scheme:dark;--bg:#111817;--panel:#1b2422;--panel2:#232f2b;--line:#34433d;--text:#f1eee3;--muted:#a8b4a9;--accent:#e7bf76;--green:#7cc5a1;--danger:#f4a49b;font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;font-synthesis:none}
* { box-sizing:border-box}
body { margin:0;color:var(--text);background:radial-gradient(ellipse at 12% 0,#29332b 0,transparent 38%),var(--bg);min-height:100vh}
button,input,select { font:inherit}
button { cursor:pointer}
button:disabled { cursor:not-allowed;opacity:.5}
button:focus-visible,input:focus-visible,select:focus-visible,a:focus-visible { outline:2px solid var(--accent);outline-offset:3px}
a { color:var(--accent)}
[hidden] { display:none!important}
/* Cabeçalho e marca. */
.topbar { border-bottom:1px solid var(--line);background:#121a18e8;display:flex;justify-content:space-between;align-items:center;gap:18px;padding:18px max(24px,calc((100vw - 1200px)/2));min-height:76px}
.brand { display:flex;align-items:center;gap:11px}
.brand-mark { display:grid;place-items:center;width:40px;height:40px;border:1px solid #b89963;color:var(--accent);background:#342f23;font-family:Georgia,serif;font-size:26px;font-weight:bold;transform:rotate(-5deg);border-radius:6px}
.brand-name { display:block;font-weight:850;letter-spacing:.15em;font-size:14px}
.brand-sub { display:block;font-size:11px;color:var(--muted);margin-top:3px}
.status { font-size:12px;color:var(--muted);border:1px solid var(--line);border-radius:999px;padding:7px 11px;white-space:nowrap;display:flex;align-items:center;gap:7px}
.status i,.small-dot { width:7px;height:7px;border-radius:50%;background:var(--accent);display:inline-block;flex-shrink:0}
.status.ready i,.small-dot { background:var(--green)}
.status.failed i { background:var(--danger)}
.shell { max-width:1248px;margin:0 auto;padding:38px 24px 20px}
.intro { display:flex;justify-content:space-between;align-items:end;gap:30px;margin-bottom:28px}
.eyebrow,.section-label { font-size:10px;letter-spacing:.17em;color:var(--accent);font-weight:750}
.eyebrow span { color:#637368}
h1 { font-family:Georgia,serif;font-size:clamp(30px,4.6vw,48px);font-weight:normal;letter-spacing:-.035em;line-height:1.08;margin:13px 0 9px}
h2 { font-size:19px;letter-spacing:-.025em;margin:7px 0 17px}
p { line-height:1.5;margin:0 0 8px}
.intro p { font-size:14px;color:var(--muted);max-width:650px;margin:0}
.stats { display:flex;align-items:stretch;flex-shrink:0;gap:1px;border:1px solid var(--line);border-radius:12px;background:var(--line);overflow:hidden}
.stats div { padding:12px 18px;background:#1c2622;min-width:83px}
.stats b { display:block;color:var(--accent);font-size:26px;font-weight:650;font-variant-numeric:tabular-nums;letter-spacing:-.045em}
.stats span { display:block;font-size:11px;color:var(--muted);white-space:nowrap}
/* Layout em duas colunas. */
.workspace { display:grid;grid-template-columns:340px minmax(0,1fr);gap:18px;align-items:stretch}
.panel { border:1px solid var(--line);border-radius:14px;background:var(--panel);min-width:0;box-shadow:0 14px 38px #0002}
.controls { padding:22px}
/* Seletores e formulário de consulta. */
.modes { display:flex;gap:5px;background:#111917;padding:4px;border:1px solid var(--line);border-radius:9px;margin-bottom:20px}
.mode { width:50%;border:0;background:transparent;padding:9px;color:var(--muted);font-weight:650;font-size:13px;border-radius:6px}
.mode.active { background:#e7bf76;color:#211d15}
.field { margin-bottom:15px;min-width:0}
.field label,legend { display:block;font-size:12px;font-weight:650;color:#dce3d6;margin-bottom:7px}
.field-grid { display:grid;grid-template-columns:1fr 1fr;gap:10px}
select,input[type=number] { width:100%;min-width:0;background:#141d1a;border:1px solid #415149;color:var(--text);border-radius:7px;padding:10px 11px;font-size:13px;min-height:40px}
select:hover,input:hover { border-color:#788d7c}
.label-row { display:flex;justify-content:space-between;align-items:center;gap:8px}
.label-row label { white-space:nowrap}
.label-row span { font-size:11px;color:var(--muted);font-weight:400}
.text-button { font-size:11px;border:0;background:none;color:var(--accent);padding:0 0 7px}
/* Indicador visual do percentual de vida. */
.health-track { height:4px;background:#0f1614;margin-top:8px;border-radius:4px;overflow:hidden}
.health-track div { background:var(--green);height:100%;width:100%;transition:width .15s}
/* Lista dos itens selecionáveis. */
.inventory { border:0;padding:0;margin:0 0 18px;min-width:0}
.inventory legend { width:100%;display:flex;justify-content:space-between;align-items:center}
.inventory legend span { color:var(--muted);font-weight:400;font-size:11px}
.items-list { border:1px solid var(--line);border-radius:8px;max-height:166px;overflow:auto;background:#141d1a;scrollbar-color:#65776d #141d1a}
.item { display:flex;gap:8px;align-items:center;padding:9px 10px;border-bottom:1px solid #2c3832;font-size:12px;cursor:pointer}
.item:last-child { border:0}
.item:hover { background:#26352e}
.item input { accent-color:var(--accent);width:15px;height:15px;margin:0;flex-shrink:0}
.item span:nth-child(2) { flex:1}
.item .cost { font-variant-numeric:tabular-nums;font-size:11px;color:var(--accent)}
.primary { display:flex;justify-content:space-between;align-items:center;width:100%;border:0;border-radius:8px;background:var(--accent);color:#272015;padding:13px 15px;font-size:13px;font-weight:800}
.primary:hover:not(:disabled) { background:#f4d395}
.match-info { font-size:13px;line-height:1.6;border:1px solid var(--line);border-radius:8px;padding:14px;background:#15201b;color:var(--muted);margin-bottom:18px}
.error { color:var(--danger);font-size:12px;margin:11px 0 0}
/* Área de conversa. */
.chat { display:flex;flex-direction:column;min-height:650px;max-height:800px}
.chat-head { padding:22px 24px 16px;display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--line);gap:12px}
.chat-head h2 { margin-bottom:0}
.secondary { font-size:12px;color:var(--muted);background:transparent;border:1px solid var(--line);border-radius:7px;padding:9px 12px;white-space:nowrap}
.secondary:hover { color:var(--text);border-color:#6c8070}
.chat-body { flex:1;overflow-y:auto;min-height:400px;padding:25px 24px 30px;background:linear-gradient(145deg,#1a2420,#17201d);scrollbar-color:#65776d #17201d}
.message { display:flex;gap:10px;margin-bottom:20px;align-items:flex-start}
.message:last-child { margin-bottom:0}
.avatar { height:30px;width:30px;flex:0 0 30px;display:grid;place-items:center;border:1px solid #796848;border-radius:9px;background:#302b20;color:var(--accent);font-weight:800;font-family:Georgia,serif}
.bubble { font-size:14px;border:1px solid #3b4a40;border-radius:3px 12px 12px 12px;background:#26322b;padding:15px 17px;max-width:min(580px,calc(100% - 40px));overflow-wrap:anywhere}
.bubble p:last-child { margin-bottom:0}
.sender { font-size:10px;font-weight:750;letter-spacing:.13em;color:var(--accent);margin-bottom:8px}
.muted { font-size:12px;color:var(--muted)}
.user { justify-content:flex-end}
.user .bubble { background:#3d3527;border-color:#736245;border-radius:12px 3px 12px 12px;max-width:88%}
.user .sender { color:#f5d79d}
.result-heading { font-size:11px;color:var(--accent);letter-spacing:.1em;text-transform:uppercase;font-weight:750;margin-top:13px}
.recommendation { font-size:20px;font-weight:750;line-height:1.3;margin:5px 0}
.metric { display:inline-block;font-size:12px;border:1px solid #53634f;color:#e8e9d9;background:#202a23;border-radius:5px;padding:4px 8px;margin:3px 0 8px;font-variant-numeric:tabular-nums}
/* Ranking das ações e seus valores Q. */
.ranking { margin-top:12px;border-top:1px solid #46564a;padding-top:8px}
.ranking summary { cursor:pointer;font-size:12px;font-weight:650;color:var(--accent);padding:5px 0}
.rank-row { display:grid;grid-template-columns:minmax(0,1fr) auto;gap:10px;padding:7px 0;font-size:12px;border-bottom:1px solid #36453a}
.rank-row:last-child { border:0}
.rank-row small { display:block;color:var(--muted);font-size:11px;margin-top:2px}
.rank-value { font-variant-numeric:tabular-nums;align-self:center;white-space:nowrap}
.chat-foot { padding:13px 24px;display:flex;align-items:center;gap:7px;color:var(--muted);font-size:11px;border-top:1px solid var(--line)}
footer { display:flex;justify-content:space-between;gap:16px;font-size:11px;color:#93a397;margin-top:20px;line-height:1.5}
/* Layout responsivo para telas menores. */
@media(max-width:850px) { .intro { flex-direction:column;align-items:flex-start;gap:20px}
.workspace { grid-template-columns:1fr}
.chat { min-height:460px;max-height:650px}
.chat-body { min-height:300px}
.stats { width:100%}
.stats div { flex:1}
.topbar { padding:14px 20px}
.shell { padding:25px 16px}
.controls { padding:19px}
footer { flex-direction:column;gap:4px}
}
@media(max-width:420px) { .stats div { padding:9px 10px;min-width:0}
.stats b { font-size:22px}
.stats span { font-size:10px}
.chat-head,.chat-body { padding:16px}
.chat-head h2 { font-size:16px}
.secondary { font-size:11px;padding:7px}
.bubble { padding:12px}
.status { font-size:10px;padding:6px 8px}
}

## 7. JavaScript — comunicação e interatividade

In [ ]:
%%writefile app.js
// A página coleta o contexto e consulta o servidor; não implementa aprendizado próprio.
const $ = (id) => document.getElementById(id);
const form = $('query-form');
let config, mode = 'item', busy = false;
const number = (v, digits = 2) => Number(v).toLocaleString('pt-BR', { maximumFractionDigits: digits });
const qNumber = (v) => Number(v).toLocaleString('pt-BR', { minimumFractionDigits: 6, maximumFractionDigits: 6 });
const initialMessage = $('messages').firstElementChild.cloneNode(true);
// Cria elementos usando textContent, sem interpretar texto como HTML.
function node(tag, className, text) {
  const el = document.createElement(tag);
  if (className) el.className = className;
  if (text !== undefined) el.textContent = text;
  return el;
}
function option(value, label) { const el = node('option', '', label); el.value = value; return el; }
// Lê os IDs dos itens marcados no formulário.
function inventory() { return [...$('items').querySelectorAll('input:checked')].map(el => Number(el.value)); }
function heroData() { return config.herois.find(h => h.nome === $('heroi').value); }
function maxHealth() { return heroData().vida + config.itens.filter(i => inventory().includes(i.id)).reduce((s, i) => s + i.vida_extra, 0); }
// Atualiza a barra de vida e limita o inventário a dois itens.
function health(fill = false) {
  if (!config) return;
  if (fill) $('vida').value = '100';
  const max = maxHealth(), pct = Number($('vida').value);
  $('vida-max').textContent = `${number(max*pct/100)} / ${number(max)} HP`;
  $('health-fill').style.width = `${pct}%`;
  const count = inventory().length;
  $('inventory-count').textContent = `${count} / ${config.max_itens} itens`;
  for (const el of $('items').querySelectorAll('input')) el.disabled = mode !== 'item' || (!el.checked && count >= config.max_itens);
}
// Impede selecionar o mesmo herói como jogador e adversário.
function updateEnemies() {
  if (!config) return;
  const previous = $('adversario').value || 'Abrams';
  const candidates = config.herois.filter(h => mode === 'heroi' || h.nome !== $('heroi').value);
  $('adversario').replaceChildren(...candidates.map(h => option(h.nome, h.nome)));
  if (candidates.some(h => h.nome === previous)) $('adversario').value = previous;
}
// Alterna entre recomendação de itens e comparação de heróis.
function setMode(value) {
  mode = value;
  const itemMode = mode === 'item';
  $('mode-item').classList.toggle('active', itemMode);
  $('mode-hero').classList.toggle('active', !itemMode);
  $('mode-item').setAttribute('aria-pressed', String(itemMode));
  $('mode-hero').setAttribute('aria-pressed', String(!itemMode));
  $('hero-field').hidden = !itemMode;
  $('item-fields').hidden = !itemMode;
  $('hero-info').hidden = itemMode;
  $('item-fields').querySelectorAll('input,select').forEach(el => el.disabled = !itemMode);
  $('heroi').disabled = !itemMode;
  $('submit').textContent = itemMode ? 'Consultar agente ↗' : 'Comparar heróis ↗';
  $('form-error').hidden = true;
  updateEnemies();
  if (itemMode) health();
}
// Adiciona uma mensagem ao histórico da conversa.
function addMessage(kind, text) {
  const message = node('div', `message ${kind}`);
  if (kind === 'bot') message.append(node('div', 'avatar', 'Q'));
  const bubble = node('div', 'bubble');
  bubble.append(node('div', 'sender', kind === 'bot' ? 'ASSISTENTE' : 'VOCÊ'));
  if (text) bubble.append(node('p', '', text));
  message.append(bubble);
  $('messages').append(message);
  $('messages').scrollTop = $('messages').scrollHeight;
  return bubble;
}
// Formata vida restante, resultado e duração do duelo.
function resumoDuelo(d) {
  const resultado = d.vitoria ? 'Vitória simulada' : d.empate ? 'Empate no duelo' : 'Derrota simulada';
  return `${resultado} · Vida final: ${number(d.vida_restante)} / ${number(d.vida_maxima)} HP · Inimigo: ${number(d.vida_inimigo)} / ${number(d.vida_max_inimigo)} HP · ${number(d.duracao)} s`;
}
// Apresenta os dados da resposta: Q, empates, diferença para guardar e ranking.
function showResult(result) {
  const bubble = addMessage('bot', result.mensagem);
  if (!result.conhecido) return;
  const item = result.tipo === 'item';
  bubble.append(node('div', 'result-heading', item ? 'Recomendação de compra' : 'Heróis recomendados · 1 duelo'));
  const tied = result.melhores.length > 1;
  const title = item ? `${result.sugerida.nome}${tied ? ' (menor custo entre empatadas)' : ''}` : result.melhores.map(r => r.nome).join(' / ');
  bubble.append(node('div', 'recommendation', title));
  if (tied) bubble.append(node('p', 'muted', item ? `${result.melhores.length} ações têm retorno equivalente na tolerância do modelo. Priorizamos a de menor custo, sem alterar os valores Q. Consulte todas no ranking.` : 'Há heróis com retorno equivalente. Veja o ranking.'));
  bubble.append(node('span', 'metric', `Melhor valor Q: ${qNumber(result.valor_q)}`));
  if (item) {
    bubble.append(node('p', '', resumoDuelo(result.sugerida.duelo)));
    bubble.append(node('p', 'muted', `Diferença de Q vs. guardar almas: ${qNumber(result.sugerida.delta_guardar)}. Chance base de acerto estimada: jogador ${number(100*result.prob_acerto,1)}%, inimigo ${number(100*result.prob_acerto_inimigo,1)}% (antes do voo).`));
  } else {
    for (const row of result.melhores) bubble.append(node('p', '', `${row.nome} — ${row.acao_sugerida}. ${resumoDuelo(row.duelo)}`));
  }
  bubble.append(node('p', 'muted', item ? `${result.acoes_conhecidas} de ${result.acoes_validas} ações válidas cobertas pelo treinamento e replay.` : `${result.herois_conhecidos} de ${result.herois_validos} heróis elegíveis têm dados para esta comparação.`));
  if (result.valor_q < 0) bubble.append(node('p', 'muted', 'Atenção: mesmo a melhor alternativa tem retorno estimado negativo. A recomendação não garante vitória.'));
  bubble.append(node('p', 'muted', 'Q combina resultado, vida restante ou dano líquido e recompensas futuras descontadas. Não é uma chance de vitória. A simulação utiliza dano esperado pela chance de acerto, não sorteia cada tiro.'));
  const details = node('details', 'ranking');
  details.append(node('summary', '', `Ver ranking (${result.ranking.length})`));
  for (const row of result.ranking) {
    const line = node('div', 'rank-row');
    const desc = node('div', '', row.nome);
    desc.append(node('small', '', item ? `${number(row.custo)} almas · ΔQ vs guardar: ${qNumber(row.delta_guardar)} · ${number(row.visitas_episodios)} visitas em episódios + ${number(row.lotes_replay)} lotes de replay` : `${row.acao_sugerida} · Chance base de acerto: ${number(100*row.prob_acerto,1)}%`));
    desc.append(node('small', '', resumoDuelo(row.duelo)));
    if (result.melhores.some(best => item ? best.id === row.id : best.nome === row.nome)) desc.append(node('small', '', 'Melhor valor Q (inclui empates)'));
    line.append(desc, node('span', 'rank-value', qNumber(row.valor_q)));
    details.append(line);
  }
  bubble.append(details);
  $('messages').scrollTop = $('messages').scrollHeight;
}
$('mode-item').addEventListener('click', () => setMode('item'));
$('mode-hero').addEventListener('click', () => setMode('heroi'));
$('heroi').addEventListener('change', () => { updateEnemies(); health(); });
$('items').addEventListener('change', () => health());
$('vida').addEventListener('change', () => health());
$('vida-cheia').addEventListener('click', () => health(true));
$('clear-chat').addEventListener('click', () => $('messages').replaceChildren(initialMessage.cloneNode(true)));
// Envia a consulta ao servidor; não executa o Q-learning no navegador.
form.addEventListener('submit', async (event) => {
  event.preventDefault();
  if (!config || busy) return;
  $('form-error').hidden = true;
  const payload = { tipo: mode, adversario: $('adversario').value, distancia: Number($('distancia').value) };
  let text = `Qual herói devo escolher contra ${payload.adversario} a ${payload.distancia} m?`;
  if (mode === 'item') {
    Object.assign(payload, { heroi: $('heroi').value, almas: Number($('almas').value), etapas_restantes: Number($('etapas').value), vida_pct: Number($('vida').value), inventario: inventory() });
    const names = config.itens.filter(i => payload.inventario.includes(i.id)).map(i => i.nome);
    text = `Qual item devo escolher para ${payload.heroi} contra ${payload.adversario} a ${payload.distancia} m? Tenho ${number(payload.almas)} almas, ${payload.vida_pct}% de vida, ${payload.etapas_restantes} duelos restantes e ${names.length ? names.join(', ') : 'nenhum item'}.`;
  }
  addMessage('user', text);
  busy = true; $('submit').disabled = true;
  try {
    const response = await fetch('/api/recomendar', {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(payload)});
    const data = await response.json();
    if (!response.ok) throw new Error(data.erro || 'Não foi possível consultar o modelo.');
    showResult(data);
  } catch (error) {
    addMessage('bot', `Não foi possível completar a consulta: ${error.message}`);
  } finally { busy = false; $('submit').disabled = false; }
});
// Carrega as opções e os metadados do modelo antes de liberar o formulário.
async function init() {
  try {
    const response = await fetch('/api/config');
    if (!response.ok) throw new Error('Servidor indisponível.');
    config = await response.json();
    $('heroi').replaceChildren(...config.herois.map(h => option(h.nome, h.nome)));
    $('distancia').replaceChildren(...config.distancias.map(d => option(d, `${d} m — ${d === 2 ? 'curto' : d === 15 ? 'médio' : 'longo'}`)));
    $('distancia').value = '15';
    $('items').replaceChildren(...config.itens.map(i => {
      const label = node('label', 'item');
      const input = node('input'); input.type = 'checkbox'; input.value = i.id;
      label.append(input, node('span', '', i.nome), node('span', 'cost', number(i.custo)));
      return label;
    }));
    $('hero-count').textContent = number(config.herois.length);
    $('item-count').textContent = number(config.itens.length);
    $('match-count').textContent = number(config.herois.length * (config.herois.length - 1) * config.distancias.length);
    $('almas').replaceChildren(...config.saldos.map(s => option(s, number(s))));
    $('almas').value = String(config.almas_iniciais);
    $('vida').replaceChildren(...config.vidas_pct.map(p => option(p, `${p}%`)));
    $('vida').value = '100';
    $('status').replaceChildren(node('i'), document.createTextNode('Modelo conectado'));
    $('status').classList.add('ready');
    $('model-info').textContent = `Q-learning · ${number(config.episodios_por_heroi * config.herois.length)} episódios · ${number(config.estados)} cenários na interface · ${number(config.cobertura_pct)}% de cobertura das ações`;
    setMode('item'); health(true); $('submit').disabled = false;
  } catch (error) {
    $('status').replaceChildren(node('i'), document.createTextNode('Modelo indisponível'));
    $('status').classList.add('failed');
    $('form-error').textContent = 'Inicie o servidor Python do projeto e abra o endereço informado no terminal. Não abra o HTML diretamente.';
    $('form-error').hidden = false;
  }
}
init();

## 8. Testar as recomendações

In [ ]:
from treinamento import carregar_modelo
from IPython.display import display

# Carrega o modelo sem executar o treinamento.
modelo = carregar_modelo()
print("Cobertura da interface (%):", metadados(modelo)["cobertura_pct"])
# Exemplo de recomendação de item: qualquer combinação permitida.
dados = dict(tipo="item", heroi="Infernus", adversario="Abrams", distancia=25,
             etapas_restantes=4, almas=3200, vida_pct=25,
             inventario=list(amb.custos_itens)[:2])
display(responder(dados, modelo))
# Exemplo de comparação entre os quatro heróis contra Abrams.
display(responder(dict(tipo="heroi", adversario="Abrams", distancia=15), modelo))

## 9. Abrir a interface (opcional)

A célula abaixo inicia o servidor na sessão do notebook. Altere `ABRIR_INTERFACE` para `True` para usá-lo. Em Jupyter local, abra `http://127.0.0.1:8000`; no Colab, a disponibilidade da janela depende das permissões do navegador e do ambiente de execução. As consultas da seção 8 funcionam diretamente nas células.

In [ ]:
ABRIR_INTERFACE = False

if ABRIR_INTERFACE:
    import threading

    # O servidor consulta o modelo já carregado e entrega a página ao navegador.
    servidor = criar_servidor(modelo, porta=8000)
    thread = threading.Thread(target=servidor.serve_forever, daemon=True)
    thread.start()
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(8000)
    except ImportError:
        print("Abra http://127.0.0.1:8000 no navegador.")
    # Para encerrar, execute servidor.shutdown() e servidor.server_close().